<a href="https://colab.research.google.com/github/jonaidsharif/Machine-Learning-Projects/blob/main/TWITTER_SENTIMENT_ANALYSIS_using_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Use drive.mount with force_remount=True to ensure the drive is remounted
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Importing the Dependencies

In [2]:
import pandas as pd

# Try reading the file with 'latin-1' encoding
df = pd.read_csv('/content/drive/MyDrive/Machine Learning/Sentiment140.csv', encoding='latin-1')

# Or try to let pandas auto-detect the encoding
# df = pd.read_csv('/content/drive/MyDrive/Machine Learning/Sentiment140.csv', encoding_errors='replace')


df.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [3]:
# import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [4]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
# printing the stopwords in English
print(stopwords.words('english'))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

## Data Collection & Preprocessing

In [6]:
# loading the data from csv file to pandas dataframe
twitter_data = pd.read_csv('/content/drive/MyDrive/Machine Learning/Sentiment140.csv', encoding='ISO-8859-1')

In [7]:
# Checking the number of rows and columns
twitter_data.shape

(1599999, 6)

In [8]:
# printing the first 5 rows of the dataflow
twitter_data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [9]:
# Naming the columns and reading the dataset again

column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv('/content/drive/MyDrive/Machine Learning/Sentiment140.csv', encoding='ISO-8859-1', names=column_names)

In [10]:
# check the number of rows and columns
twitter_data.shape

(1600000, 6)

In [11]:
# print the first 5 rows of the dataframe
twitter_data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [12]:
# counting the number of missing values in the dataset
twitter_data.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [13]:
# checking the distribution of target column
twitter_data['target'].value_counts()

,count
target,
0,800000
4,800000


#  Convert the target '4' to '1'

In [14]:
twitter_data.replace({'target': {4: 1}}, inplace=True)

In [15]:
# checking the distribution of target column
twitter_data['target'].value_counts()

,count
target,
0,800000
1,800000


0 which is the Negative Tweet

1 which is the Positive Tweet

**Stemming** meaning is the process of reducing a word to its Root word

Example: actor, actress, acting = act

In [16]:
port_stem = PorterStemmer()

In [18]:
def steming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()

  return stemmed_content

In [19]:
twitter_data['text'] = twitter_data['text'].apply(steming)

In [20]:
twitter_data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"[switchfoot, http, twitpic, com, y, zl, awww, ..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,"[is, upset, that, he, can, t, update, his, fac..."
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,"[kenichan, i, dived, many, times, for, the, ba..."
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,"[my, whole, body, feels, itchy, and, like, its..."
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"[nationwideclass, no, it, s, not, behaving, at..."


In [21]:
print(twitter_data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


In [22]:
# Separating the data and label

X = twitter_data['text']
Y = twitter_data['target']

In [24]:
print(X)

0          [switchfoot, http, twitpic, com, y, zl, awww, ...
1          [is, upset, that, he, can, t, update, his, fac...
2          [kenichan, i, dived, many, times, for, the, ba...
3          [my, whole, body, feels, itchy, and, like, its...
4          [nationwideclass, no, it, s, not, behaving, at...
                                 ...                        
1599995    [just, woke, up, having, no, school, is, the, ...
1599996    [thewdb, com, very, cool, to, hear, old, walt,...
1599997    [are, you, ready, for, your, mojo, makeover, a...
1599998    [happy, th, birthday, to, my, boo, of, alll, t...
1599999    [happy, charitytuesday, thenspcc, sparkscharit...
Name: text, Length: 1600000, dtype: object


In [25]:
print(Y)

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


S

# Spliting the data to training data and test



In [27]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [28]:
print(X.shape, X_train.shape, X_test.shape)

(1600000,) (1280000,) (320000,)


In [29]:
print(X_train)

1570269    [about, to, watch, saw, iv, and, drink, a, lil...
1273074                            [hatermagazine, i, m, in]
88479      [even, though, its, my, favourite, drink, i, t...
254604     [i, think, my, hands, got, burnt, in, the, sun...
667941     [took, mazie, to, the, dr, for, shots, today, ...
                                 ...                        
941805                                  [threewinks, cheers]
1007131    [i, vote, for, livewire, to, play, live, smith...
1460311                  [is, eager, for, monday, afternoon]
929226     [hope, everyone, and, their, mother, had, a, g...
526253     [i, love, waking, up, to, folgers, too, bad, m...
Name: text, Length: 1280000, dtype: object


In [30]:
print(X_test)

131348     [mmangen, m, doing, fine, i, haven, t, had, mu...
1142114    [at, ahs, may, show, w, ruth, kim, amp, geoffr...
244564     [ishatara, maybe, it, was, only, a, bay, area,...
445353              [game, just, ended, they, lost, stinkyy]
415893               [i, am, not, as, cool, as, my, brother]
                                 ...                        
178459     [this, twitter, is, driving, me, nuts, wont, l...
1515130                [teamqivana, you, re, welcome, again]
1449952    [destini, nevertheless, hooray, for, members, ...
441063                             [not, feeling, too, well]
1583304                            [supersandro, thank, you]
Name: text, Length: 320000, dtype: object


In [34]:
# Converting the textual data to numerical data

vectorizer = TfidfVectorizer()

# Fit the vectorizer to the training data and then transform it
X_train = vectorizer.fit_transform(X_train.apply(' '.join)) # Join the list of strings into a single string

# Transform the test data using the fitted vectorizer
X_test = vectorizer.transform(X_test.apply(' '.join)) # Join the list of strings into a single string

In [35]:
print(X_train)

  (0, 2067)	0.24193877001313313
  (0, 453363)	0.12148485475980697
  (0, 481939)	0.30447551653830146
  (0, 390656)	0.3293975539873257
  (0, 206332)	0.4857480076242065
  (0, 17690)	0.1496211864872621
  (0, 121501)	0.38436070312484305
  (0, 258884)	0.3855833943595694
  (0, 488987)	0.41322347757019917
  (1, 178952)	0.9765387376484952
  (1, 199337)	0.21534180706932754
  (2, 453363)	0.06900936442712897
  (2, 17690)	0.0849921828093998
  (2, 121501)	0.43667151573506924
  (2, 138282)	0.16663525388625475
  (2, 449007)	0.15950386355270021
  (2, 205309)	0.2834707637326193
  (2, 308705)	0.16759876713928085
  (2, 144082)	0.25070038846383297
  (2, 448054)	0.28650103709365976
  (2, 442727)	0.14215717985609486
  (2, 478415)	0.281805932852511
  (2, 85757)	0.2679936085012617
  (2, 442340)	0.10230075002928594
  (2, 489389)	0.3537352115264888
  :	:
  (1279998, 103958)	0.18542347355285077
  (1279998, 175223)	0.21336006149095368
  (1279998, 188630)	0.22920711233364058
  (1279998, 485504)	0.20346683466366844


In [36]:
print(X_test)

  (0, 16490)	0.14905337456526022
  (0, 34100)	0.14105809855616946
  (0, 74655)	0.24267729958055045
  (0, 117558)	0.18127349461543463
  (0, 117923)	0.37255794382515695
  (0, 147438)	0.22171903098840867
  (0, 151763)	0.09918070262208285
  (0, 154087)	0.20542529369520834
  (0, 175223)	0.14639330766721614
  (0, 179213)	0.20184426616769688
  (0, 190757)	0.2444262495958155
  (0, 204032)	0.09622033057314465
  (0, 297665)	0.39309733733236507
  (0, 306361)	0.1544897058630911
  (0, 308705)	0.08825846288186953
  (0, 330981)	0.10700243608606022
  (0, 428462)	0.1911527571697233
  (0, 440221)	0.36075534212580534
  (0, 442727)	0.07486077848826732
  (0, 451067)	0.2809252422136495
  (0, 453363)	0.1453628933614196
  (0, 464356)	0.15809690753437994
  (1, 7848)	0.5300122919686152
  (1, 16490)	0.18433110824098314
  (1, 28891)	0.14811632285349516
  :	:
  (319995, 460676)	0.24165933724246727
  (319995, 464356)	0.20116353727238379
  (319995, 491054)	0.28034427836460685
  (319996, 6550)	0.290606337222887
  (31

# Training the ML Model

Logistic Regression

In [37]:
model = LogisticRegression(max_iter=1000) # maximun number of time

In [38]:
model.fit(X_train, Y_train)

LogisticRegression(max_iter=1000)

Model Evaluation

Accuracy score

In [39]:
# Accuracy score on the training data

X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [40]:
print('Accuracy score on the training data: ', training_data_accuracy)

Accuracy score on the training data:  0.79999609375


In [41]:
# Accuracy score on the test data

X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [42]:
print('Accuracy score on the testing data: ', test_data_accuracy)

Accuracy score on the testing data:  0.79535625


Model accuracy = 79.5%

Saving the trained model

In [43]:
import pickle

In [44]:
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))

Using the saved model for future predictions

In [46]:
# loading the saved model
loaded_model = pickle.load(open('trained_model.sav', 'rb'))

In [49]:
# Check the length of Y_test to ensure it has at least 201 items
print(len(Y_test))

# Print some of the indices of Y_test to understand its structure
print(Y_test.index)

# If you want to access the 201st item, use iloc
print(Y_test.iloc[200])

# Alternatively, if you know the actual index value you want to access, use loc
# print(Y_test.loc[index_value])

X_new = X_test[200]

# Now you can access the element based on how you accessed it above
# Replace the code below with one of the options
# print(Y_test[200]) # Only if 200 is a valid index
print(Y_test.iloc[200]) # If you want the 201st item
# print(Y_test.loc[index_value]) # If you know the actual index value


prediction = model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print('The news is Negative Tweet')
else:
  print('The news is Positive Tweet')

320000
Index([ 131348, 1142114,  244564,  445353,  415893,  311439,  823188,  738844,
        285768,  248234,
       ...
        617422,   40225,  190322,  779708,  359621,  178459, 1515130, 1449952,
        441063, 1583304],
      dtype='int64', length=320000)
1
1
[1]
The news is Positive Tweet


In [50]:
X_new = X_test[300]

prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print('The news is Negative Tweet')
else:
  print('The news is Positive Tweet')

[0]
The news is Negative Tweet
